In [7]:
from decouple import config
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import pandas as pd


## Criar conexao

In [8]:
POSTGRES_NAME = config('POSTGRES_NAME')
POSTGRES_USER = config('POSTGRES_USER')
POSTGRES_PASSWORD = config('POSTGRES_PASSWORD')
POSTGRES_HOST = config('POSTGRES_HOST')
POSTGRES_PORT = config('POSTGRES_PORT')

       
DATABASE_URL = f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_NAME}"

# Criando a engine do SQLAlchemy
engine = create_engine(DATABASE_URL)

## Carregamento dos Dados no SQL


In [9]:
link = 'https://github.com/silva-fabiofreitas/Estatistica_python/raw/main/dados/IDH.xlsx'
df = pd.read_excel(link, sheet_name='Base')
df.head()

,Espacialidades,ND,EV,Mi,PS60,QL,PIB,RP,Gi,P,VP,FC18,SC25,EAE,T18,PDA,PDBA,PDCL
0,Angra dos Reis,Outros,75.75,12.97,83.19,1.426171,4.540722e+06,798.68,0.50,6.69,21.42,55.41,7.42,9.00,5.43,92.49,95.45,99.26
1,Aperibé,C.Vicioso,72.10,18.40,77.85,1.113099,4.077564e+04,516.14,0.43,9.40,29.72,51.36,5.94,8.95,11.06,96.84,99.52,97.07
2,Araruama,Outros,75.32,14.18,82.88,0.409663,5.643963e+05,680.88,0.54,11.60,32.86,55.57,10.27,8.84,7.84,96.08,94.42,95.64
3,Areal,C.Vicioso,74.35,15.00,81.22,0.326421,8.742154e+04,571.74,0.48,11.13,32.21,46.76,6.24,9.21,7.77,85.66,98.20,98.57
4,Armação dos Búzios,C.Vicioso,74.44,14.80,81.36,0.054656,5.751335e+05,851.39,0.51,3.69,17.24,58.03,11.25,9.09,4.83,83.53,96.31,98.55


In [10]:
# Carregar dados no banco
df.to_sql('idh_data', con=engine, if_exists='replace', index=False)

-1

In [11]:
db = SQLDatabase(engine=engine)
print(db.dialect)
print(db.get_usable_table_names())
print(db.run("SELECT COUNT(*) FROM idh_data LIMIT 2;"))
print(db.run("SELECT * FROM idh_data LIMIT 2;"))
print(db.get_table_info(['idh_data']))

postgresql
['auth_group', 'auth_group_permissions', 'auth_permission', 'auth_user', 'auth_user_groups', 'auth_user_user_permissions', 'core_category', 'core_iris', 'core_mindmap', 'django_admin_log', 'django_content_type', 'django_migrations', 'django_session', 'idh_data']
[(92,)]
[('Angra dos Reis', 'Outros', 75.75, 12.97, 83.19, 1.4261707521102645, 4540721.53301557, 798.68, 0.5, 6.69, 21.42, 55.41, 7.42, 9.0, 5.43, 92.49, 95.45, 99.26), ('Aperibé', 'C.Vicioso', 72.1, 18.4, 77.85, 1.1130994852579565, 40775.6429497641, 516.14, 0.43, 9.4, 29.72, 51.36, 5.94, 8.95, 11.06, 96.84, 99.52, 97.07)]

CREATE TABLE idh_data (
	"Espacialidades" TEXT, 
	"ND" TEXT, 
	"EV" DOUBLE PRECISION, 
	"Mi" DOUBLE PRECISION, 
	"PS60" DOUBLE PRECISION, 
	"QL" DOUBLE PRECISION, 
	"PIB" DOUBLE PRECISION, 
	"RP" DOUBLE PRECISION, 
	"Gi" DOUBLE PRECISION, 
	"P" DOUBLE PRECISION, 
	"VP" DOUBLE PRECISION, 
	"FC18" DOUBLE PRECISION, 
	"SC25" DOUBLE PRECISION, 
	"EAE" DOUBLE PRECISION, 
	"T18" DOUBLE PRECISION, 
	"PDA

## LLM

In [23]:
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [13]:
from langchain import hub

query_prompt_template = hub.pull("langchain-ai/sql-query-system-prompt")

assert len(query_prompt_template.messages) == 1
query_prompt_template.messages[0].pretty_print()

/home/fabiofreitas/PycharmProjects/WEB_IA/.venv/lib/python3.12/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


================================ System Message ================================

Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

Question: {input}


# Gerarar query

In [ ]:
from typing_extensions import Annotated, TypedDict

class State(TypedDict):
    question: str
    query: str
    result: str
    answer: str


# class QueryOutput(TypedDict):
#     """Generated SQL query."""

#     query: Annotated[str, ..., "Syntactically valid SQL query."]

from pydantic import BaseModel

class QueryOutput(BaseModel):
    query =


def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(['idh_data']),
            "input": state["question"],
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

write_query({"question": "Quantidade de municipios por ND"})

/home/fabiofreitas/PycharmProjects/WEB_IA/.venv/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(
/home/fabiofreitas/PycharmProjects/WEB_IA/.venv/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(


BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'tools[0].function.description': string too long. Expected a string with maximum length 1024, but got a string with length 1824 instead.", 'type': 'invalid_request_error', 'param': 'tools[0].function.description', 'code': 'string_above_max_length'}}